In [1]:
import numpy as np
import networkx as nx
import pandas as pd
from math import log
from collections import defaultdict

# ============================================================
# DATASET
# ============================================================

np.random.seed(42)

n = 10
m = 8
lam = 3
mu = 3

S = np.random.uniform(0.2, 0.9, (n, m))

# Critical paper (paper 0)
S[:, 0] = 0.15
S[0, 0] = 0.95
S[1, 0] = 0.88

print("Similarity Matrix")
print(np.round(S, 2))


# ============================================================
# HELPERS
# ============================================================

def assignment_values(assignment, S, m):
    vals = []

    for p in range(m):
        total = 0.0
        for r, paper in assignment:
            if paper == p:
                total += S[r, p]
        vals.append(total)

    return vals


def maxmin_fairness(assignment, S, m):
    vals = assignment_values(assignment, S, m)
    return min(vals)


def nsw_score(assignment, S, m):
    vals = assignment_values(assignment, S, m)

    eps = 1e-9
    prod = 1.0

    for v in vals:
        prod *= max(v, eps)

    return prod ** (1 / m)


# ============================================================
# PEERREVIEW4ALL SUBROUTINE
# ============================================================

def flow_assignment(reviewers,
                    papers,
                    capacities,
                    paper_need,
                    S):

    pairs = []

    for r in reviewers:
        for p in papers:
            pairs.append((S[r, p], r, p))

    pairs.sort(reverse=True)

    active_edges = []

    for sim, r, p in pairs:

        active_edges.append((r, p))

        G = nx.DiGraph()

        source = "s"
        sink = "t"

        for r0 in reviewers:
            G.add_edge(source,
                       f"r{r0}",
                       capacity=capacities[r0])

        for p0 in papers:
            G.add_edge(f"p{p0}",
                       sink,
                       capacity=paper_need)

        for rr, pp in active_edges:
            G.add_edge(f"r{rr}",
                       f"p{pp}",
                       capacity=1)

        flow_val, flow_dict = nx.maximum_flow(G, source, sink)

        if flow_val == len(papers) * paper_need:

            assignment = []

            for rr in reviewers:
                node = f"r{rr}"

                if node not in flow_dict:
                    continue

                for paper_node, f in flow_dict[node].items():
                    if f > 0:
                        paper = int(paper_node[1:])
                        assignment.append((rr, paper))

            return assignment

    return None


# ============================================================
# PEERREVIEW4ALL
# ============================================================

def peerreview4all(S, lam=3, mu=3):

    n, m = S.shape

    remaining_papers = set(range(m))
    capacities = {r: mu for r in range(n)}

    final_assignment = []

    while remaining_papers:

        candidates = []

        for kappa in range(1, lam + 1):

            assn = flow_assignment(
                reviewers=list(range(n)),
                papers=list(remaining_papers),
                capacities=capacities,
                paper_need=kappa,
                S=S
            )

            if assn is not None:
                candidates.append(assn)

        if not candidates:
            break

        best = max(
            candidates,
            key=lambda A: maxmin_fairness(A, S, m)
        )

        vals = {}

        for p in remaining_papers:
            vals[p] = sum(
                S[r, p]
                for r, pp in best
                if pp == p
            )

        worst = min(vals.values())

        worst_papers = [
            p for p, v in vals.items()
            if abs(v - worst) < 1e-9
        ]

        for p in worst_papers:

            chosen = [
                (r, pp)
                for r, pp in best
                if pp == p
            ]

            final_assignment.extend(chosen)

            for r, _ in chosen:
                capacities[r] -= 1

        remaining_papers -= set(worst_papers)

    return final_assignment


# ============================================================
# NSW MIN COST FLOW
# ============================================================

def nsw_min_cost_flow(S,
                      lam=3,
                      mu=3,
                      threshold=0.5):

    n, m = S.shape

    B = (S >= threshold).astype(int)

    G = nx.DiGraph()

    source = "s"
    sink = "t"

    demand = m * lam

    G.add_node(source, demand=-demand)
    G.add_node(sink, demand=demand)

    # reviewer supply
    for r in range(n):

        G.add_edge(
            source,
            f"r{r}",
            capacity=mu,
            weight=0
        )

    # binary expertise edges
    for r in range(n):
        for p in range(m):

            if B[r, p] == 1:
                G.add_edge(
                    f"r{r}",
                    f"p{p}",
                    capacity=1,
                    weight=0
                )

    # diminishing returns
    BIG = 10000

    for p in range(m):

        prev = 1

        for k in range(1, lam + 1):

            marginal = log(k + 1) - log(k)

            cost = int(-1000 * marginal)

            G.add_edge(
                f"p{p}",
                sink,
                capacity=1,
                weight=cost
            )

    flow = nx.min_cost_flow(G)

    assignment = []

    for r in range(n):

        node = f"r{r}"

        for pnode, f in flow[node].items():

            if pnode.startswith("p") and f > 0:

                p = int(pnode[1:])

                assignment.append((r, p))

    return assignment


# ============================================================
# RUN
# ============================================================

pr4a_assignment = peerreview4all(S, lam, mu)
nsw_assignment = nsw_min_cost_flow(S, lam, mu)

pr4a_vals = assignment_values(pr4a_assignment, S, m)
nsw_vals = assignment_values(nsw_assignment, S, m)

comparison = pd.DataFrame({
    "Metric": [
        "NSW",
        "Max-Min Fairness"
    ],
    "PeerReview4All": [
        nsw_score(pr4a_assignment, S, m),
        min(pr4a_vals)
    ],
    "Binary NSW": [
        nsw_score(nsw_assignment, S, m),
        min(nsw_vals)
    ]
})

print("\nComparison")
print(comparison)

print("\nPeerReview4All valuations")
print(sorted(np.round(pr4a_vals, 3)))

print("\nNSW valuations")
print(sorted(np.round(nsw_vals, 3)))

Similarity Matrix
[[0.95 0.87 0.71 0.62 0.31 0.31 0.24 0.81]
 [0.88 0.7  0.21 0.88 0.78 0.35 0.33 0.33]
 [0.15 0.57 0.5  0.4  0.63 0.3  0.4  0.46]
 [0.15 0.75 0.34 0.56 0.61 0.23 0.63 0.32]
 [0.15 0.86 0.88 0.77 0.41 0.27 0.68 0.51]
 [0.15 0.55 0.22 0.84 0.38 0.66 0.42 0.56]
 [0.15 0.33 0.88 0.74 0.86 0.83 0.62 0.85]
 [0.15 0.34 0.23 0.43 0.47 0.39 0.78 0.45]
 [0.15 0.58 0.3  0.76 0.25 0.89 0.74 0.34]
 [0.15 0.77 0.69 0.71 0.74 0.25 0.45 0.28]]


NetworkXUnfeasible: no flow satisfies all node demands